# 44. CD-OPE-S Stage-wise Feature 분석

red 성능 개선이 stage-1 color separability 감소와 연결되는지 검증합니다.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch4_utils.py").exists():
    matches = list(Path.cwd().glob("Deeplearning/*/4장/ch4_utils.py")) + list(Path.cwd().glob("**/ch4_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "4장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch4_utils import *

paths = find_ch4_paths()
set_korean_font()
set_seed(41)
paths

Chapter4Paths(chapter4_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장'), chapter3_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/manifests'), design_path=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/chapter4_cd_ope_s_architecture_design.md'), validity_path=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/chapter4_cd_ope_s_design_validity_review.md'))

## 44-1. feature bank 추출

In [2]:
manifests = create_ch4_manifests(max_per_cell=3, seed=41)
seed_metrics = collect_ch4_cd_ope_metrics()
if seed_metrics.empty:
    raise FileNotFoundError("42번에서 CD-OPE-S 학습 run을 먼저 생성하세요.")

TARGET_VARIANT = "gl_cd_consistency_style"
TARGET_SEED = 0
selected = seed_metrics[
    (seed_metrics["variant"] == TARGET_VARIANT)
    & (seed_metrics["model_seed"] == TARGET_SEED)
]
if selected.empty:
    raise FileNotFoundError(f"선택한 run이 없습니다: {TARGET_VARIANT} seed {TARGET_SEED}")
run_dir = Path(selected.iloc[0]["run_dir"])

feature_out = paths.runs_root / "cd_ope_s" / "feature_bank" / TARGET_VARIANT / f"seed_{TARGET_SEED}"
feature_paths = extract_feature_bank_for_cd_ope_s(
    run_dir,
    manifests["eval_matched_probe"],
    feature_out,
    batch_size=8,
    max_samples=120,
    seed=41,
)
feature_paths

C:\Users\준승\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'features': WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/cd_ope_s/feature_bank/gl_cd_consistency_style/seed_0/stage_features.npz'),
 'index': WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/cd_ope_s/feature_bank/gl_cd_consistency_style/seed_0/sample_index.csv'),
 'energy': WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/cd_ope_s/feature_bank/gl_cd_consistency_style/seed_0/stage_energy.csv')}

## 44-2. separability 계산

In [3]:
sep_out = paths.runs_root / "cd_ope_s" / "feature_separability" / TARGET_VARIANT / f"seed_{TARGET_SEED}"
sep = summarize_feature_separability(feature_paths["features"], feature_paths["index"], sep_out)
display(sep.sort_values("nearest_centroid_accuracy", ascending=False).head(30))

color = sep[sep["label"] == "color_group"].sort_values("feature")
defect = sep[sep["label"] == "defect_type"].sort_values("feature")
display(color)
display(defect)

,feature,label,n_samples,n_labels,nearest_centroid_accuracy,mean_pairwise_centroid_distance
14,stage2_global,shape_group,120,2,1.000000,0.406637
38,stage4_global,shape_group,120,2,1.000000,1.464609
34,stage3_background,shape_group,120,2,1.000000,0.549803
22,stage2_background,shape_group,120,2,1.000000,0.405030
26,stage3_global,shape_group,120,2,0.976190,0.546605
46,stage4_background,shape_group,120,2,0.976190,1.494979
10,stage1_background,shape_group,120,2,0.928571,0.221725
2,stage1_global,shape_group,120,2,0.928571,0.218815
25,stage3_global,defect_type,120,4,0.900000,0.693748
37,stage4_global,defect_type,120,4,0.825000,0.798150


,feature,label,n_samples,n_labels,nearest_centroid_accuracy,mean_pairwise_centroid_distance
8,stage1_background,color_group,120,5,0.800,0.405749
0,stage1_global,color_group,120,5,0.800,0.406998
4,stage1_target,color_group,120,5,0.425,0.727711
20,stage2_background,color_group,120,5,0.400,0.219198
12,stage2_global,color_group,120,5,0.350,0.229359
16,stage2_target,color_group,120,5,0.350,1.539867
32,stage3_background,color_group,120,5,0.150,0.248420
24,stage3_global,color_group,120,5,0.150,0.268050
28,stage3_target,color_group,120,5,0.200,1.937848
44,stage4_background,color_group,120,5,0.100,0.303733


,feature,label,n_samples,n_labels,nearest_centroid_accuracy,mean_pairwise_centroid_distance
9,stage1_background,defect_type,120,4,0.175,0.091499
1,stage1_global,defect_type,120,4,0.225,0.082737
5,stage1_target,defect_type,120,4,0.700,0.988889
21,stage2_background,defect_type,120,4,0.350,0.135966
13,stage2_global,defect_type,120,4,0.575,0.210382
17,stage2_target,defect_type,120,4,0.825,4.314280
33,stage3_background,defect_type,120,4,0.825,0.487619
25,stage3_global,defect_type,120,4,0.900,0.693748
29,stage3_target,defect_type,120,4,0.725,7.248052
45,stage4_background,defect_type,120,4,0.750,0.705969


## 44-3. baseline과 비교

In [4]:
baseline_path = paths.chapter3_dir / "runs" / "feature_separability" / "stage_feature_separability.csv"
if baseline_path.exists():
    baseline = pd.read_csv(baseline_path)
    comp = sep.merge(
        baseline[["feature", "label", "nearest_centroid_accuracy"]].rename(
            columns={"nearest_centroid_accuracy": "baseline_accuracy"}
        ),
        on=["feature", "label"],
        how="left",
    )
    comp["delta_vs_baseline"] = comp["nearest_centroid_accuracy"] - comp["baseline_accuracy"]
    display(comp[comp["label"].isin(["color_group", "defect_type"])].sort_values(["label", "feature"]))
else:
    print("3장 baseline separability 파일이 없습니다.")

,feature,label,n_samples,n_labels,nearest_centroid_accuracy,mean_pairwise_centroid_distance,baseline_accuracy,delta_vs_baseline
8,stage1_background,color_group,120,5,0.800,0.405749,0.858824,-0.058824
0,stage1_global,color_group,120,5,0.800,0.406998,0.894118,-0.094118
4,stage1_target,color_group,120,5,0.425,0.727711,0.647059,-0.222059
20,stage2_background,color_group,120,5,0.400,0.219198,0.752941,-0.352941
12,stage2_global,color_group,120,5,0.350,0.229359,0.764706,-0.414706
16,stage2_target,color_group,120,5,0.350,1.539867,0.552941,-0.202941
32,stage3_background,color_group,120,5,0.150,0.248420,0.494118,-0.344118
24,stage3_global,color_group,120,5,0.150,0.268050,0.505882,-0.355882
28,stage3_target,color_group,120,5,0.200,1.937848,0.317647,-0.117647
44,stage4_background,color_group,120,5,0.100,0.303733,0.423529,-0.323529
